In [1]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["kopi_nusantara"]
koleksi_menu = db["menu"]
print("Koneksi ke MongoDB berhasil!")

Koneksi ke MongoDB berhasil!


In [2]:
#A. Membangun Katalog (Create)
koleksi_menu.delete_many({})

koleksi_menu.insert_many([
    {"nama_menu": "Espresso", "harga": 18000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 100},
    {"nama_menu": "Americano", "harga": 20000, "kategori": "Kopi", "tingkat_kepahitan": 4, "stok": 70},
    {"nama_menu": "Kopi Susu Gula Aren", "harga": 24000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 120},
    {"nama_menu": "Cappuccino", "harga": 25000, "kategori": "Kopi", "tingkat_kepahitan": 3, "stok": 70},
    {"nama_menu": "Caffe Latte", "harga": 21000, "kategori": "Kopi", "tingkat_kepahitan": 2, "stok": 80},
    {"nama_menu": "Kopi Tubruk", "harga": 10000, "kategori": "Kopi", "tingkat_kepahitan": 5, "stok": 100},
    {"nama_menu": "Roti Bakar", "harga": 10000, "kategori": "Makanan", "pedas_level": 0, "stok": 120},
    {"nama_menu": "Kentang Goreng", "harga": 15000, "kategori": "Makanan", "pedas_level": 0, "stok": 60},
    {"nama_menu": "Pisang Goreng", "harga": 10000, "kategori": "Makanan", "pedas_level": 0, "stok": 75},
    {"nama_menu": "Nasi Goreng ", "harga": 20000, "kategori": "Makanan", "pedas_level": 3, "stok": 35},
    {"nama_menu": "Mie Goreng Spesial", "harga": 28000, "kategori": "Makanan", "pedas_level": 3, "stok": 30},
    {"nama_menu": "Ayam Geprek", "harga": 20000, "kategori": "Makanan", "pedas_level": 3, "stok": 30},
    {"nama_menu": "Mendoan", "harga": 10000, "kategori": "Makanan", "pedas_level": 0, "stok": 85},
])
print("Jumlah dokumen di collection 'menu':", koleksi_menu.count_documents({}))

Jumlah dokumen di collection 'menu': 13


In [3]:
#B. Kueri Bertingkat (Read)
print("1. Kopi dengan tingkat_kepahitan > 3:")
for m in koleksi_menu.find({"kategori": "Kopi", "tingkat_kepahitan": {"$gt": 3}}):
    print(f"  - {m['nama_menu']} (kepahitan={m['tingkat_kepahitan']})")

print("\n2. Harga antara 20000 dan 25000:")
for m in koleksi_menu.find({"harga": {"$gte": 20000, "$lte": 25000}}):
    print(f"  - {m['nama_menu']} (Rp{m['harga']:,})")

print("\n3. Tanpa field pedas_level:")
for m in koleksi_menu.find({"pedas_level": {"$exists": False}}):
    print(f"  - {m['nama_menu']} ({m['kategori']})")

1. Kopi dengan tingkat_kepahitan > 3:
  - Espresso (kepahitan=5)
  - Americano (kepahitan=4)
  - Kopi Tubruk (kepahitan=5)

2. Harga antara 20000 dan 25000:
  - Americano (Rp20,000)
  - Kopi Susu Gula Aren (Rp24,000)
  - Cappuccino (Rp25,000)
  - Caffe Latte (Rp21,000)
  - Nasi Goreng  (Rp20,000)
  - Ayam Geprek (Rp20,000)

3. Tanpa field pedas_level:
  - Espresso (Kopi)
  - Americano (Kopi)
  - Kopi Susu Gula Aren (Kopi)
  - Cappuccino (Kopi)
  - Caffe Latte (Kopi)
  - Kopi Tubruk (Kopi)


In [4]:
#C. Pembaruan Data (Update)
koleksi_menu.update_many({}, {"$set": {"status": "Tersedia"}})
koleksi_menu.update_one({"nama_menu": "Americano"}, {"$set": {"status": "Habis"}})
koleksi_menu.update_one({"nama_menu": "Ayam Geprek"}, {"$set": {"status": "Habis"}})

for m in koleksi_menu.find({}, {"_id": 0, "nama_menu": 1, "status": 1}):
    print(f"  - {m['nama_menu']}: {m['status']}")

  - Espresso: Tersedia
  - Americano: Habis
  - Kopi Susu Gula Aren: Tersedia
  - Cappuccino: Tersedia
  - Caffe Latte: Tersedia
  - Kopi Tubruk: Tersedia
  - Roti Bakar: Tersedia
  - Kentang Goreng: Tersedia
  - Pisang Goreng: Tersedia
  - Nasi Goreng : Tersedia
  - Mie Goreng Spesial: Tersedia
  - Ayam Geprek: Habis
  - Mendoan: Tersedia


In [5]:
#D. Integrasi ke Spark
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import avg, count
import pandas as pd

spark = SparkSession.builder.appName("Tugas9").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df_pandas = pd.DataFrame(list(koleksi_menu.find({})))

def ke_str(x): return None if pd.isna(x) else str(x)
def ke_int(x): return None if pd.isna(x) else int(x)

baris = [(ke_str(d.get("_id")), ke_str(d.get("nama_menu")), ke_str(d.get("kategori")),
          ke_int(d.get("harga")), ke_int(d.get("stok")),
          ke_int(d.get("tingkat_kepahitan")), ke_int(d.get("pedas_level")),
          ke_str(d.get("status"))) for d in df_pandas.to_dict("records")]

skema = StructType([
    StructField("_id", StringType(), True),
    StructField("nama_menu", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("harga", LongType(), True),
    StructField("stok", LongType(), True),
    StructField("tingkat_kepahitan", LongType(), True),
    StructField("pedas_level", LongType(), True),
    StructField("status", StringType(), True),
])

df_spark_menu = spark.createDataFrame(baris, schema=skema)
df_spark_menu.printSchema()
df_spark_menu.show(truncate=False)

df_spark_menu.groupBy("kategori").agg(
    avg("harga").alias("rata_rata_harga"),
    count("nama_menu").alias("jumlah_menu")
).show()

spark.stop()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/31 14:40:17 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


root
 |-- _id: string (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- harga: long (nullable = true)
 |-- stok: long (nullable = true)
 |-- tingkat_kepahitan: long (nullable = true)
 |-- pedas_level: long (nullable = true)
 |-- status: string (nullable = true)



+------------------------+-------------------+--------+-----+----+-----------------+-----------+--------+
|_id                     |nama_menu          |kategori|harga|stok|tingkat_kepahitan|pedas_level|status  |
+------------------------+-------------------+--------+-----+----+-----------------+-----------+--------+
|6ae5fbd5555a41538805e514|Espresso           |Kopi    |18000|100 |5                |NULL       |Tersedia|
|6ae5fbd5555a41538805e515|Americano          |Kopi    |20000|70  |4                |NULL       |Habis   |
|6ae5fbd5555a41538805e516|Kopi Susu Gula Aren|Kopi    |24000|120 |2                |NULL       |Tersedia|
|6ae5fbd5555a41538805e517|Cappuccino         |Kopi    |25000|70  |3                |NULL       |Tersedia|
|6ae5fbd5555a41538805e518|Caffe Latte        |Kopi    |21000|80  |2                |NULL       |Tersedia|
|6ae5fbd5555a41538805e519|Kopi Tubruk        |Kopi    |10000|100 |5                |NULL       |Tersedia|
|6ae5fbd5555a41538805e51a|Roti Bakar         |

+--------+------------------+-----------+
|kategori|   rata_rata_harga|jumlah_menu|
+--------+------------------+-----------+
|    Kopi|19666.666666666668|          6|
| Makanan|16142.857142857143|          7|
+--------+------------------+-----------+



#E. Kesimpulan
Tantangan utamanya itu adalah ketidaksamaan antar file dokumen. Seperti contoh kopi cuma punya tingkat_kepahitan, lalu makanan hanya punya pedas_level, padahal tadi spark butuh tabel yang beragam. Cara mengatasinya denga  mengubah _id menjadi string, mengganti NaN menjadi None agar menjadi null di Spark, lalu mengembalikan kolom angka menjadi integer, lalu mendefinisikan skema secara eksplisit dengan StructType dan nullable = True. Dengan begitu kolomnya menjadi seragam, field yang tidak dimiliki suatu menu tersimpan sebagai null, dan groupBy tetap berjalan benar. Masalah lain muncul saat data dimasukkan ke pandas, karena field yang tidak ada berubah menjadi NaN dan kolom bilangan bulat ikut menjadi float, misalnya 5 menjadi 5.0. Selain itu, _id bertipe ObjectId tidak dikenali oleh Spark.

In [7]:
from pymongo import MongoClient
client = MongoClient("mongodb://localhost:27017/")
koleksi_produk = client["praktikum_bigdata"]["produk"]
# $in: nilai cocok dengan salah satu isi list
list(koleksi_produk.find({"kategori": {"$in": ["Snack", "Minuman"]}}))

# $or: salah satu kondisi terpenuhi
list(koleksi_produk.find({"$or": [{"harga": {"$lt": 10000}}, {"stok": {"$gt": 100}}]}))

# $ne: tidak sama dengan
list(koleksi_produk.find({"kategori": {"$ne": "Makanan"}}))

# $regex: pencarian pola teks (nama diawali huruf K)
list(koleksi_produk.find({"nama_produk": {"$regex": "^K"}}))

# Mencari di dalam array: produk yang tags-nya mengandung "manis"
list(koleksi_produk.find({"tags": "manis"}))

[{'_id': ObjectId('6ae5db903f9620570c4f4d22'),
  'nama_produk': 'Kue',
  'kategori': 'Snack',
  'harga': 30000,
  'stok': 20,
  'tags': ['manis'],
  'diskon': 2000},
 {'_id': ObjectId('6ae5db903f9620570c4f4d23'),
  'nama_produk': 'Stroberi',
  'kategori': 'Minuman',
  'harga': 10000,
  'stok': 40,
  'tags': ['dingin', 'manis'],
  'diskon': 2000}]

In [8]:
# Stok berkurang 5 setelah terjual
koleksi_produk.update_one({"nama_produk": "Kripik"}, {"$inc": {"stok": -5}})

# Menambah tag baru
koleksi_produk.update_one({"nama_produk": "Kripik"}, {"$push": {"tags": "pedas"}})

# Menghapus field diskon dari semua dokumen
koleksi_produk.update_many({}, {"$unset": {"diskon": ""}})

# Upsert: Teh Tarik belum ada, jadi dibuat baru
koleksi_produk.update_one(
    {"nama_produk": "Teh Tarik"},
    {"$set": {"kategori": "Minuman", "harga": 14000, "stok": 30}},
    upsert=True
)

UpdateResult({'n': 1, 'upserted': ObjectId('6ae61718f004a3302a8e53b3'), 'nModified': 0, 'ok': 1.0, 'updatedExisting': False}, acknowledged=True)

In [9]:
pipeline = [
    {"$match": {"stok": {"$gt": 0}}},                    # 1. saring
    {"$group": {                                          # 2. kelompokkan
        "_id": "$kategori",
        "rata_harga": {"$avg": "$harga"},
        "jumlah_produk": {"$sum": 1},
        "total_stok": {"$sum": "$stok"}
    }},
    {"$sort": {"rata_harga": -1}}                         # 3. urutkan
]

for hasil in koleksi_produk.aggregate(pipeline):
    print(hasil)

{'_id': 'Snack', 'rata_harga': 17500.0, 'jumlah_produk': 2, 'total_stok': 105}
{'_id': 'Makanan', 'rata_harga': 17000.0, 'jumlah_produk': 4, 'total_stok': 180}
{'_id': 'Minuman', 'rata_harga': 15000.0, 'jumlah_produk': 5, 'total_stok': 470}


In [10]:
koleksi_produk.create_index("harga")
print(koleksi_produk.index_information())

# Lihat cara MongoDB menjalankan kueri
rencana = koleksi_produk.find({"harga": {"$lt": 20000}}).explain()
print(rencana["queryPlanner"]["winningPlan"])

{'_id_': {'v': 2, 'key': [('_id', 1)]}, 'harga_1': {'v': 2, 'key': [('harga', 1)]}}
{'isCached': False, 'stage': 'FETCH', 'inputStage': {'stage': 'IXSCAN', 'keyPattern': {'harga': 1}, 'indexName': 'harga_1', 'isMultiKey': False, 'multiKeyPaths': {'harga': []}, 'isUnique': False, 'isSparse': False, 'isPartial': False, 'indexVersion': 2, 'direction': 'forward', 'indexBounds': {'harga': ['[-inf.0, 20000)']}}}


In [11]:
dok = koleksi_produk.find_one()
print(dok["_id"])
print("Dibuat pada:", dok["_id"].generation_time)

6ae5db843f9620570c4f4d1c
Dibuat pada: 2026-10-31 12:13:56+00:00


In [12]:
koleksi_produk.delete_one({"nama_produk": "Kue"})   # satu dokumen
koleksi_produk.delete_many({"stok": 0})             # banyak dokumen
# koleksi_produk.drop()                              # hapus seluruh collection
# client.drop_database("kopi_nusantara")             # hapus seluruh database

DeleteResult({'n': 0, 'ok': 1.0}, acknowledged=True)